In [17]:
import os
import sys 
os.chdir("/workspaces/dev")
sys.path.append("/workspaces/dev/modules")

In [18]:
from RTWhisper import SentenceStreamer, Hyperparameters
import librosa
import numpy as np

In [19]:
MODEL_SIZE = "large-v3"

SAMPLE_RATE = 16000
BUFFER_SIZE = 10

In [20]:
audio, sr = librosa.load("/workspaces/dev/.data/funny_show.mp3", sr=SAMPLE_RATE)

In [1]:
# audio = audio[85 * SAMPLE_RATE:]

In [22]:
total_samples = len(audio)

segments = []
pos = 0
while pos < total_samples:
  rand_len = int(np.random.normal(loc=16000, scale=400))
  rand_len = np.clip(rand_len, 14000, 18000)
  end = min(pos + rand_len, total_samples)

  chunk = audio[pos:end]
  segments.append(chunk)
  pos = end

In [23]:
# full_text = ""
# for segment in segments:
#   if len(segment) < 160:
#     continue
#   seg, info = whisper.translate(segment, language="ko")
#   for s in seg:
#     full_text += s.text

In [24]:
# print(full_text)

In [25]:
HYPERPARAMETERS = {
  "sentence_max_prev_sentence": 0,
  "weighted_and_offset_token_boundary": 8000,
  "duration_filter_z": {
    "default": 2.0,
    "ko": 2.0,
    "en": 2.0,
  },
  "probability_filter": {
    "z": {
      "default": 2.0,
      "ko": 2.0,
      "en": 2.0,
    },
    "min_prob": {
      "default": 1.0,
      "ko": 0.4,
      "en": 0.4,
    },
  },
  "selector": {
    "search_range_sc": {
      "default": 24000,
      "ko": 24000,
      "en": 24000,
    },
    "threshold": {
      "default": 0.5,
      "ko": 0.25,
      "en": 0.5,
    },
    "padding": {
      "default": 3200,
      "ko": 3200,
      "en": 3200,
    },
    "tolerance": {
      "default": 8000,
      "ko": 8000,
      "en": 8000,
    },
  },
  "classifier_max_prev_sc": {
    "default": 48000
  }
}

In [26]:
hyper = Hyperparameters(None, HYPERPARAMETERS)

In [27]:
whisper_service = SentenceStreamer.get_instance(hyper)

In [28]:
raise Exception("stop")

Exception: stop

In [29]:
from RTWhisper.data import Param
from IPython.display import Audio

In [32]:
segment_id = 0
completed = {}
param = Param()

In [31]:
segment = segments[segment_id]
segment_id += 1

param.audio = segment

result = whisper_service.process(param)
completed.update(result.completed)

print(f"{segment_id}" + "--" * 20)
print([(v.lang, v.text) for k, v in completed.items()])
print(result.prev_sentence)
print([(v.lang, v.text) for v in result.prev_recog])

param.update(result)

Audio(result.prev_processed_audio, rate=SAMPLE_RATE)

1----------------------------------------
[]
None
[('ko', ' 우리'), ('ko', ' 회사에서')]


In [ ]:
Audio(result.prev_audio, rate=SAMPLE_RATE)

In [33]:
for segment in segments:
  param.audio = segment

  result = whisper_service.process(param)
  completed.update(result.completed)

  print(f"{segment_id}" + "--" * 20)
  print([(v.lang, v.text) for k, v in completed.items()])
  print(result.prev_sentence)
  print([(v.lang, v.text) for v in result.prev_recog])

  param.update(result)

0----------------------------------------
[]
None
[('ko', ' 우리'), ('ko', ' 회사에서')]
0----------------------------------------
[]
None
[('ko', ' 우리'), ('ko', ' 회사에서'), ('ko', ' 나한테'), ('ko', ' 뭐라고')]
0----------------------------------------
[]
None
[('ko', ' 우리'), ('ko', ' 회사에서'), ('ko', ' 나한테'), ('ko', ' 뭐라고'), ('ko', ' 날'), ('ko', ' 설득을'), ('ko', ' 할까')]
0----------------------------------------
[(['ko'], '우리 회사에서 나한테 뭐라고 날 설득을 할까')]
['ko'] 우리 회사에서 나한테 뭐라고 날 설득을 할까
[('ko', ' 그걸'), ('ko', ' 한번'), ('ko', ' 들어보고'), ('ko', ' 싶어요.')]
0----------------------------------------
[(['ko'], '우리 회사에서 나한테 뭐라고 날 설득을 할까')]
['ko'] 우리 회사에서 나한테 뭐라고 날 설득을 할까
[('ko', ' 그걸'), ('ko', ' 한번'), ('ko', ' 들어보고'), ('ko', ' 싶었던'), ('ko', ' 거야.')]
0----------------------------------------
[(['ko'], '우리 회사에서 나한테 뭐라고 날 설득을 할까')]
['ko'] 우리 회사에서 나한테 뭐라고 날 설득을 할까
[('ko', ' 그걸'), ('ko', ' 한번'), ('ko', ' 들어보고'), ('ko', ' 싶었던'), ('ko', ' 거야')]
0----------------------------------------
[(['ko'], '우리 회사에서 나한테 뭐라고 날 설득을 할까')

In [34]:
for key, item in completed.items():
  print(key, item)
for v in result.prev_recog:
  print(v.text)
# for v in prev_recog:
#   print(v.text)

0 ['ko'] 우리 회사에서 나한테 뭐라고 날 설득을 할까
1 ['ko'] 그걸 한 번 들어보고 싶었던 거야.
2 ['ko'] 어 근데 잡을 생각조차 없는 거야
3 ['ko'] 본인들끼리 또 지은이가 다른 데 좋은데 가고 싶다고 하면 우리가 어떻게 잡냐 이런 식으로 지금 회사 입장에서 이야기를 끝내셨다는 거야
4 ['ko'] 그래서 왜 그렇게 말씀을 하시냐고 내가 그랬어.
5 ['ko'] 왜 그렇게 말씀하세요?
6 ['ko'] 돈만 있으면 되잖아요
7 ['ko'] 왜 돈이 있단 말을 안 해요?
8 ['ko'] 왜 말을 못해?
9 ['ko'] 이들만큼 있으면 뭔 소리를 하는 거야.
10 ['ko'] 너무 사고 싶을 수도 있잖아요.
11 ['ko'] 이러면서 오히려 마음을 알게 된 계기가 됐죠.
12 ['ko'] 너 눈치 없이 재계약한 거 아니야?
13 ['ko'] 그쪽에서 원할 게 아니야.
14 ['ko'] 신인 가수가 2만 6백만 원이야.
 괜찮아.
